## 04_memory_games_cleaning.ipynb
- i have 5 files/csv that i will be cleaning up
- i will utilize the data cleaning function that claude helped me come up with in 03_gamelog_cleaning.ipynb in order to clean these csv as fast and efficient as possible

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import psycopg2
from pandasql import sqldf
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
pd.options.plotting.backend = 'plotly'
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
import voila
import warnings
warnings.filterwarnings('ignore')

In [7]:
def clean_gamelog(file_path,season,game_type):
    df = pd.read_csv(file_path)

    # 1. remove the totals row (the only row with no team game number)
    df = df[df['Gtm'].notna()].copy()

    # 2. status: 'Played', or the reason he didn't play (Gcar is blank when he didn't play)
    # Gcar stands for career game number
    df['status'] = np.where(df['Gcar'].isna(), df['GS'], 'Played')

    # started: Yes if GS is '*', otherwise No
    # GS stands for games started. Lebron has always mostly started since he started his career
    df['started'] = np.where(df['GS'] == '*', 'Yes', 'No')
    df = df.drop(columns=['GS'])

    # 3. home_away: '@' means away, blank means home
    df = df.rename(columns={'Unnamed: 5': 'home_away'})
    df['home_away'] = np.where(df['home_away'] == '@', 'Away', 'Home')

    # 4. split Result ('L 80-88') into win_loss ('L') and score ('80-88')
    df['win_loss'] = df['Result'].str[0]
    df['score'] = df['Result'].str[2:]
    df = df.drop(columns=['Result'])

    # 5. MP: '42:39' -> 42.65 decimal minutes (missed games become empty)
    parts = df['MP'].str.split(':', expand=True)
    minutes = pd.to_numeric(parts[0], errors='coerce')
    seconds = pd.to_numeric(parts[1], errors='coerce')
    df['MP'] = (minutes + seconds / 60).round(2)

    # 6. Date: '10/26/10' -> a real date (2010-10-26)
    df['Date'] = pd.to_datetime(df['Date'], format='mixed')

    # 7. stat columns to numbers ('Did Not Play' text becomes empty)
    text_cols = ['Date', 'Team', 'home_away', 'Opp', 'win_loss', 'score', 'status', 'started', 'game_type']
    num_cols = [col for col in df.columns if col not in text_cols]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

    # 8. SQL-friendly column names
    df.columns = (
        df.columns
        .str.lower()
        .str.replace('%', '_pct')
        .str.replace('3p', 'three_p')
        .str.replace('2p', 'two_p')
        .str.replace('+/-', 'plus_minus')
    )

    # 9. add the season and gametype
    df['season'] = season
    df['game_type'] = game_type

    return df

In [8]:
p12 = clean_gamelog('../data/raw/lebron_gamelog_playoffs_2012.csv', '2011-12', 'Playoffs')
p13 = clean_gamelog('../data/raw/lebron_gamelog_playoffs_2013.csv', '2012-13', 'Playoffs')
p16 = clean_gamelog('../data/raw/lebron_gamelog_playoffs_2016.csv', '2015-16', 'Playoffs')
p18 = clean_gamelog('../data/raw/lebron_gamelog_playoffs_2018.csv', '2017-18', 'Playoffs')
r23 = clean_gamelog('../data/raw/lebron_gamelog_2022_23.csv', '2022-23', 'Regular')

memory_games = pd.concat([p12, p13, p16, p18, r23], ignore_index=True)
memory_games.shape

(171, 38)

In [9]:
# saving the csv to my lebron_memory_gamelogs.csv
# putting this in my 'processed data' folder
memory_games.to_csv('../data/processed/lebron_memory_gamelogs.csv', index=False)

In [10]:
memory_games

,rk,gcar,gtm,date,team,home_away,opp,mp,fg,fga,...,pf,pts,gmsc,plus_minus,status,started,win_loss,score,season,game_type
0,1.0,93.0,1.0,2012-04-28,MIA,Home,NYK,31.68,10.0,14.0,...,2.0,32.0,29.9,35.0,Played,Yes,W,100-67,2011-12,Playoffs
1,2.0,94.0,2.0,2012-04-30,MIA,Home,NYK,40.58,8.0,18.0,...,1.0,19.0,16.4,10.0,Played,Yes,W,104-94,2011-12,Playoffs
2,3.0,95.0,3.0,2012-05-03,MIA,Away,NYK,35.83,9.0,21.0,...,5.0,32.0,19.2,23.0,Played,Yes,W,87-70,2011-12,Playoffs
3,4.0,96.0,4.0,2012-05-06,MIA,Away,NYK,41.45,9.0,21.0,...,4.0,27.0,13.9,-7.0,Played,Yes,L,87-89,2011-12,Playoffs
4,5.0,97.0,5.0,2012-05-09,MIA,Home,NYK,39.63,7.0,16.0,...,2.0,29.0,25.4,6.0,Played,Yes,W,106-94,2011-12,Playoffs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,51.0,1417.0,78.0,2023-04-02,LAL,Away,HOU,29.35,8.0,18.0,...,1.0,18.0,19.4,23.0,Played,Yes,W,134-109,2022-23,Regular
167,52.0,1418.0,79.0,2023-04-04,LAL,Away,UTA,38.47,14.0,27.0,...,2.0,37.0,25.3,-7.0,Played,Yes,W,135-133 (OT),2022-23,Regular
168,53.0,1419.0,80.0,2023-04-05,LAL,Away,LAC,35.10,13.0,20.0,...,0.0,33.0,26.4,-10.0,Played,Yes,L,118-125,2022-23,Regular
169,54.0,1420.0,81.0,2023-04-07,LAL,Home,PHO,29.35,6.0,19.0,...,1.0,16.0,5.3,11.0,Played,Yes,W,121-107,2022-23,Regular


In [11]:
memory_games.info()

<class 'pandas.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 38 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   rk           171 non-null    float64       
 1   gcar         144 non-null    float64       
 2   gtm          171 non-null    float64       
 3   date         171 non-null    datetime64[us]
 4   team         171 non-null    str           
 5   home_away    171 non-null    str           
 6   opp          171 non-null    str           
 7   mp           144 non-null    float64       
 8   fg           144 non-null    float64       
 9   fga          144 non-null    float64       
 10  fg_pct       144 non-null    float64       
 11  three_p      144 non-null    float64       
 12  three_pa     144 non-null    float64       
 13  three_p_pct  143 non-null    float64       
 14  two_p        144 non-null    float64       
 15  two_pa       144 non-null    float64       
 16  two_p_pct    144 no